# MLP Viseme Model Training

Upgrades the phoneme-to-blendshape model from ridge regression (linear) to a 2-layer MLP.
This addresses the regression-to-mean problem inherent to linear models, producing crisper
mouth poses (CodeTalker, CVPR 2023).

**Architecture:** `D -> 128 -> 64 -> 55` (ReLU hidden layers, sigmoid output)

**Works on:**
- Google Colab (GPU) via the VSCode Colab extension
- Local machine with `pip install -r requirements.txt`

**Usage:**
1. Open this notebook in VSCode
2. Select Kernel -> Colab -> GPU runtime (or use a local Python kernel)
3. Run all cells
4. Download the exported `.npz` model and copy to `<inference-project>/models/phoneme.npz`

## Cell 1: Environment Setup

In [ ]:
import sys
import os

# Detect environment — the VSCode Colab extension registers google.colab
# in sys.modules even locally, so check for a Colab-only env var instead.
IN_COLAB = os.environ.get("COLAB_RELEASE_TAG") is not None

if IN_COLAB:
    # Colab: torch + numpy are pre-installed, just need matplotlib
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    print("Running on Google Colab")
else:
    print("Running locally")
    print("Ensure you ran: pip install -r requirements.txt")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import json
import gzip
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 2: Data Loading

Tries to import from `src/` (works when running locally from the `viseme-training/` directory).
Falls back to inlined functions (works on Colab without uploading the full repo).

In [ ]:
NUM_WEIGHTS = 55

# ---------------------------------------------------------------------------
# Resolve the viseme-training root directory regardless of where the
# notebook kernel's cwd is (VSCode often uses the workspace root).
# ---------------------------------------------------------------------------

# Find viseme-training/ root by walking up from this notebook's location,
# or from cwd if the notebook path isn't available.
def _find_training_root():
    """Find the viseme-training/ directory."""
    # Try: relative to cwd
    for candidate in [".", "viseme-training"]:
        path = os.path.join(candidate, "data", "datasets")
        if os.path.isdir(path):
            return os.path.abspath(candidate)
    # Try: walk up from cwd
    d = os.path.abspath(".")
    for _ in range(5):
        check = os.path.join(d, "viseme-training", "data", "datasets")
        if os.path.isdir(check):
            return os.path.join(d, "viseme-training")
        check2 = os.path.join(d, "data", "datasets")
        if os.path.isdir(check2):
            return d
        d = os.path.dirname(d)
    return "."

TRAINING_ROOT = _find_training_root()
print(f"Training root: {TRAINING_ROOT}")
print(f"Working dir:   {os.getcwd()}")

# ---------------------------------------------------------------------------
# Data loading (inlined from src/data_format.py for Colab compatibility)
# ---------------------------------------------------------------------------

def load_dataset(path: str) -> dict:
    """Load a .json.gz training dataset. Returns dict with metadata + sentences."""
    if path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            return json.load(f)
    else:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)


def _find_nearest_frame(frame_times, target_ms):
    """Binary search for nearest frame to target_ms."""
    if not frame_times:
        return None
    lo, hi = 0, len(frame_times) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if frame_times[mid][0] < target_ms:
            lo = mid + 1
        else:
            hi = mid
    best_idx = lo
    best_dist = abs(frame_times[lo][0] - target_ms)
    if lo > 0:
        dist = abs(frame_times[lo - 1][0] - target_ms)
        if dist < best_dist:
            best_idx = lo - 1
    return frame_times[best_idx][1]


def get_all_labeled_frames(data: dict) -> list:
    """Extract (phoneme, target_weights) pairs from dataset."""
    all_pairs = []
    for sentence in data.get("sentences", []):
        timeline = sentence.get("phoneme_timeline", [])
        frames = sentence.get("target_frames", [])
        if not timeline or not frames:
            continue
        frame_times = [(f["time_ms"], f["weights"]) for f in frames]
        for i, phoneme in enumerate(timeline):
            phone = phoneme.get("phone", "sil")
            start_s = phoneme.get("start_s", 0.0)
            duration_s = phoneme.get("duration_s", 0.0)
            mid_ms = (start_s + duration_s / 2.0) * 1000.0
            nearest_weights = _find_nearest_frame(frame_times, mid_ms)
            if nearest_weights is None:
                continue
            prev_phone = timeline[i - 1].get("phone", "sil") if i > 0 else "sil"
            next_phone = timeline[i + 1].get("phone", "sil") if i < len(timeline) - 1 else "sil"
            if i == 0:
                position = "start"
            elif i == len(timeline) - 1:
                position = "end"
            else:
                position = "mid"
            all_pairs.append({
                "phone": phone,
                "prev_phone": prev_phone,
                "next_phone": next_phone,
                "duration_s": duration_s,
                "position": position,
                "target_weights": nearest_weights,
            })
    return all_pairs


# ---------------------------------------------------------------------------
# Featurization (inlined from src/phoneme_model.py)
# ---------------------------------------------------------------------------

def build_vocabulary(pairs: list) -> list:
    """Build sorted phoneme vocabulary from training pairs."""
    phones = set()
    for p in pairs:
        phones.add(p["phone"])
        phones.add(p["prev_phone"])
        phones.add(p["next_phone"])
    return sorted(phones)


def featurize(pairs: list, vocab: list) -> np.ndarray:
    """Convert phoneme pairs to feature matrix (N, D)."""
    phone_to_idx = {p: i for i, p in enumerate(vocab)}
    v = len(vocab)
    d = 3 * v + 4  # curr + prev + next + duration + position(3)

    durations = [p["duration_s"] for p in pairs]
    mean_dur = float(np.mean(durations)) if durations else 0.08

    X = np.zeros((len(pairs), d), dtype=np.float32)
    for i, p in enumerate(pairs):
        idx = phone_to_idx.get(p["phone"])
        if idx is not None:
            X[i, idx] = 1.0
        idx = phone_to_idx.get(p["prev_phone"])
        if idx is not None:
            X[i, v + idx] = 1.0
        idx = phone_to_idx.get(p["next_phone"])
        if idx is not None:
            X[i, 2 * v + idx] = 1.0
        X[i, 3 * v] = p["duration_s"] / mean_dur if mean_dur > 0 else 1.0
        if p["position"] == "start":
            X[i, 3 * v + 1] = 1.0
        elif p["position"] == "end":
            X[i, 3 * v + 2] = 1.0
        else:
            X[i, 3 * v + 3] = 1.0
    return X, mean_dur


def extract_targets(pairs: list) -> np.ndarray:
    """Extract target weight matrix (N, 55)."""
    Y = np.zeros((len(pairs), NUM_WEIGHTS), dtype=np.float32)
    for i, p in enumerate(pairs):
        weights = p["target_weights"]
        Y[i, :len(weights)] = weights[:NUM_WEIGHTS]
    return Y


# ---------------------------------------------------------------------------
# Load and prepare data
# ---------------------------------------------------------------------------

# Dataset candidates (relative to TRAINING_ROOT)
DATASET_NAMES = [
    "data/datasets/beat_v1.json.gz",
    "data/datasets/harvard_azure_disambig.json.gz",
    "data/datasets/harvard_azure.json.gz",
]

dataset_path = None
for name in DATASET_NAMES:
    candidate = os.path.join(TRAINING_ROOT, name)
    if os.path.isfile(candidate):
        dataset_path = candidate
        print(f"Found dataset: {dataset_path}")
        break

# If no local file found and we're on Colab, prompt upload
if dataset_path is None and IN_COLAB:
    from google.colab import files
    print("No local dataset found. Upload your training dataset (.json.gz):")
    print("  Recommended: beat_v1.json.gz (30MB, best quality)")
    print("  Alternative: harvard_azure_disambig.json.gz (7.7MB)")
    uploaded = files.upload()
    if uploaded:
        dataset_path = list(uploaded.keys())[0]
        print(f"Using uploaded: {dataset_path}")

if dataset_path is None:
    raise FileNotFoundError(
        f"No dataset found. Searched in: {TRAINING_ROOT}/data/datasets/\n"
        f"Expected one of: {DATASET_NAMES}\n"
        f"Current working directory: {os.getcwd()}"
    )

print(f"\nLoading: {dataset_path}")
data = load_dataset(dataset_path)
meta = data.get("metadata", {})
print(f"Provider: {meta.get('provider', '?')}, Sentences: {meta.get('num_sentences', 0)}, Frames: {meta.get('total_frames', 0)}")

# Extract pairs and featurize
all_pairs = get_all_labeled_frames(data)
print(f"Extracted {len(all_pairs)} phoneme instances")

vocab = build_vocabulary(all_pairs)
print(f"Vocabulary: {len(vocab)} phonemes")

# Deterministic split (same as phoneme_model.py)
rng = np.random.RandomState(42)
indices = rng.permutation(len(all_pairs))
split = int(0.85 * len(indices))
train_pairs = [all_pairs[i] for i in indices[:split]]
val_pairs = [all_pairs[i] for i in indices[split:]]

X_train, mean_duration = featurize(train_pairs, vocab)
Y_train = extract_targets(train_pairs)
X_val, _ = featurize(val_pairs, vocab)
Y_val = extract_targets(val_pairs)

D = X_train.shape[1]
print(f"\nTrain: {X_train.shape[0]}, Val: {X_val.shape[0]}")
print(f"Features: {D} dims, Output: {NUM_WEIGHTS} channels")
print(f"Mean duration: {mean_duration:.4f}s")

## Cell 3: Ridge Regression Baseline

Train the existing linear model as a comparison baseline.

In [ ]:
# Mouth channel indices for focused evaluation (16 channels that matter for lip sync)
MOUTH_CHANNELS = {
    "jawOpen": 17, "mouthClose": 18, "mouthFunnel": 19, "mouthPucker": 20,
    "mouthSmileLeft": 23, "mouthSmileRight": 24,
    "mouthStretchLeft": 29, "mouthStretchRight": 30,
    "mouthRollLower": 31, "mouthRollUpper": 32,
    "mouthPressLeft": 35, "mouthPressRight": 36,
    "mouthLowerDownLeft": 37, "mouthLowerDownRight": 38,
    "mouthUpperUpLeft": 39, "mouthUpperUpRight": 40,
}


def evaluate(Y_true, Y_pred, label=""):
    """Evaluate predictions and print results. Returns metrics dict."""
    errors = np.abs(Y_true - Y_pred)
    channel_mae = errors.mean(axis=0)

    mouth_maes = {name: float(channel_mae[idx]) for name, idx in MOUTH_CHANNELS.items()}
    avg_mouth_mae = np.mean([channel_mae[idx] for idx in MOUTH_CHANNELS.values()])
    avg_all_mae = float(errors.mean())
    quality = max(0.0, 1.0 - avg_mouth_mae)

    if label:
        print(f"\n  {label}:")
        print(f"    Avg mouth MAE: {avg_mouth_mae:.4f}")
        print(f"    Avg all MAE:   {avg_all_mae:.4f}")
        print(f"    Quality score: {quality:.3f}")

    return {
        "avg_mouth_mae": float(avg_mouth_mae),
        "avg_all_mae": avg_all_mae,
        "quality_score": float(quality),
        "channel_mae": channel_mae,
        "mouth_maes": mouth_maes,
    }


# Train ridge regression (closed-form)
alpha = 1.0
N, D_feat = X_train.shape
X_bias = np.hstack([X_train, np.ones((N, 1), dtype=np.float32)])
XtX = X_bias.T @ X_bias
reg = alpha * np.eye(D_feat + 1, dtype=np.float32)
reg[-1, -1] = 0.0
W_full = np.linalg.solve(XtX + reg, X_bias.T @ Y_train)

W_ridge = W_full[:-1, :].T  # (55, D)
b_ridge = W_full[-1, :]      # (55,)

# Evaluate ridge
Y_pred_ridge_train = np.clip(X_train @ W_ridge.T + b_ridge, 0, 1)
Y_pred_ridge_val = np.clip(X_val @ W_ridge.T + b_ridge, 0, 1)

ridge_train_metrics = evaluate(Y_train, Y_pred_ridge_train, "Ridge (train)")
ridge_val_metrics = evaluate(Y_val, Y_pred_ridge_val, "Ridge (val)")

gap = ridge_val_metrics["avg_mouth_mae"] - ridge_train_metrics["avg_mouth_mae"]
print(f"    Train/val gap: {gap:+.4f}")

## Cell 4: MLP Model Definition

Two hidden layers with ReLU, sigmoid output for smooth [0, 1] constraint.
Captures non-linear coarticulation that ridge regression misses.

In [ ]:
class VisemeMLP(nn.Module):
    """Phoneme-to-blendshape MLP: D -> hidden1 -> hidden2 -> 55."""

    def __init__(self, input_dim, hidden1=128, hidden2=64, output_dim=55):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, output_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


model = VisemeMLP(input_dim=D).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"MLP architecture: {D} -> 128 -> 64 -> 55")
print(f"Total parameters: {total_params:,}")
print(f"Ridge parameters: {W_ridge.shape[0] * W_ridge.shape[1] + W_ridge.shape[0]:,}")
print(f"Data/param ratio: {len(train_pairs) / total_params:.1f}:1")

## Cell 5: Training Loop

Adam optimizer with weight decay (L2 regularization, similar to ridge).
Early stopping monitors validation loss.

In [ ]:
# Hyperparameters
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 512
MAX_EPOCHS = 200
PATIENCE = 15

# DataLoaders
train_ds = TensorDataset(
    torch.from_numpy(X_train).to(device),
    torch.from_numpy(Y_train).to(device),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

X_val_t = torch.from_numpy(X_val).to(device)
Y_val_t = torch.from_numpy(Y_val).to(device)

# Optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

# Training
train_losses = []
val_losses = []
best_val_loss = float("inf")
best_state = None
patience_counter = 0

print(f"Training: lr={LEARNING_RATE}, wd={WEIGHT_DECAY}, batch={BATCH_SIZE}, patience={PATIENCE}")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'Val MAE':>10} {'Status':>10}")
print(f"{'-' * 55}")

t0 = time.time()

for epoch in range(MAX_EPOCHS):
    # Train
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for X_batch, Y_batch in train_loader:
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, Y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    avg_train_loss = epoch_loss / n_batches

    # Validate
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_loss = criterion(val_pred, Y_val_t).item()
        val_mae = torch.abs(Y_val_t - val_pred).mean().item()

    train_losses.append(avg_train_loss)
    val_losses.append(val_loss)

    # Early stopping
    status = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        status = "* best"
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"{epoch + 1:>6} {avg_train_loss:>12.6f} {val_loss:>12.6f} {val_mae:>10.4f} {'STOP':>10}")
            break

    if (epoch + 1) % 10 == 0 or epoch == 0 or status:
        print(f"{epoch + 1:>6} {avg_train_loss:>12.6f} {val_loss:>12.6f} {val_mae:>10.4f} {status:>10}")

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.1f}s ({epoch + 1} epochs)")

# Restore best weights
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)
    print(f"Restored best model (val loss = {best_val_loss:.6f})")

## Cell 6: Evaluation & Comparison

Side-by-side comparison of ridge regression vs MLP on validation data.

In [ ]:
# Get MLP predictions
model.eval()
with torch.no_grad():
    Y_pred_mlp_val = model(X_val_t).cpu().numpy()
    Y_pred_mlp_train = model(torch.from_numpy(X_train).to(device)).cpu().numpy()

mlp_train_metrics = evaluate(Y_train, Y_pred_mlp_train, "MLP (train)")
mlp_val_metrics = evaluate(Y_val, Y_pred_mlp_val, "MLP (val)")

gap = mlp_val_metrics["avg_mouth_mae"] - mlp_train_metrics["avg_mouth_mae"]
print(f"    Train/val gap: {gap:+.4f}")

# Comparison summary
print(f"\n{'=' * 50}")
print(f"  COMPARISON SUMMARY")
print(f"{'=' * 50}")
print(f"  {'Metric':<25} {'Ridge':>10} {'MLP':>10} {'Change':>10}")
print(f"  {'-' * 55}")

for metric_name, key in [("Avg mouth MAE", "avg_mouth_mae"), ("Avg all MAE", "avg_all_mae"), ("Quality score", "quality_score")]:
    r = ridge_val_metrics[key]
    m = mlp_val_metrics[key]
    pct = ((m - r) / r) * 100 if r != 0 else 0
    print(f"  {metric_name:<25} {r:>10.4f} {m:>10.4f} {pct:>+9.1f}%")

print(f"\n  Per mouth channel MAE:")
print(f"  {'Channel':<25} {'Ridge':>10} {'MLP':>10} {'Change':>10}")
print(f"  {'-' * 55}")

for name in sorted(MOUTH_CHANNELS.keys()):
    r = ridge_val_metrics["mouth_maes"][name]
    m = mlp_val_metrics["mouth_maes"][name]
    pct = ((m - r) / r) * 100 if r != 0 else 0
    marker = " <<<" if pct < -10 else ""
    print(f"  {name:<25} {r:>10.4f} {m:>10.4f} {pct:>+9.1f}%{marker}")

## Cell 7: Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Training/validation loss curves
ax = axes[0, 0]
ax.plot(train_losses, label="Train", alpha=0.8)
ax.plot(val_losses, label="Val", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training Loss Curves")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Per-channel MAE comparison (top 10 channels)
ax = axes[0, 1]
channels_sorted = sorted(MOUTH_CHANNELS.keys(),
                         key=lambda c: ridge_val_metrics["mouth_maes"][c], reverse=True)[:10]
ridge_maes = [ridge_val_metrics["mouth_maes"][c] for c in channels_sorted]
mlp_maes = [mlp_val_metrics["mouth_maes"][c] for c in channels_sorted]

x_pos = np.arange(len(channels_sorted))
width = 0.35
ax.barh(x_pos + width/2, ridge_maes, width, label="Ridge", color="#4C72B0")
ax.barh(x_pos - width/2, mlp_maes, width, label="MLP", color="#DD8452")
ax.set_yticks(x_pos)
ax.set_yticklabels(channels_sorted, fontsize=8)
ax.set_xlabel("MAE")
ax.set_title("Per-Channel MAE (Top 10)")
ax.legend()
ax.grid(True, alpha=0.3, axis="x")

# Plot 3: Predicted vs gold (jawOpen)
ax = axes[1, 0]
jaw_idx = MOUTH_CHANNELS["jawOpen"]
ax.scatter(Y_val[:, jaw_idx], Y_pred_ridge_val[:, jaw_idx],
           alpha=0.15, s=5, label="Ridge", color="#4C72B0")
ax.scatter(Y_val[:, jaw_idx], Y_pred_mlp_val[:, jaw_idx],
           alpha=0.15, s=5, label="MLP", color="#DD8452")
ax.plot([0, 0.5], [0, 0.5], "k--", alpha=0.3, label="Perfect")
ax.set_xlabel("Gold Standard")
ax.set_ylabel("Predicted")
ax.set_title("jawOpen: Predicted vs Gold")
ax.legend(markerscale=4)
ax.grid(True, alpha=0.3)

# Plot 4: Predicted vs gold (mouthFunnel)
ax = axes[1, 1]
funnel_idx = MOUTH_CHANNELS["mouthFunnel"]
ax.scatter(Y_val[:, funnel_idx], Y_pred_ridge_val[:, funnel_idx],
           alpha=0.15, s=5, label="Ridge", color="#4C72B0")
ax.scatter(Y_val[:, funnel_idx], Y_pred_mlp_val[:, funnel_idx],
           alpha=0.15, s=5, label="MLP", color="#DD8452")
ax.plot([0, 0.4], [0, 0.4], "k--", alpha=0.3, label="Perfect")
ax.set_xlabel("Gold Standard")
ax.set_ylabel("Predicted")
ax.set_title("mouthFunnel: Predicted vs Gold")
ax.legend(markerscale=4)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mlp_vs_ridge.png", dpi=150)
plt.show()
print("Saved: mlp_vs_ridge.png")

## Cell 8: Export to .npz

Extracts PyTorch weights as numpy arrays and saves in the same `.npz` format
used by `an inference module`, with a `model_type` key to
distinguish MLP from ridge models.

**To deploy:** copy the exported file to `<inference-project>/models/phoneme.npz`

In [ ]:
# Extract weights from PyTorch model
state = model.cpu().state_dict()

# net.0 = Linear(D, 128), net.2 = Linear(128, 64), net.4 = Linear(64, 55)
W1 = state["net.0.weight"].numpy()  # (128, D)
b1 = state["net.0.bias"].numpy()    # (128,)
W2 = state["net.2.weight"].numpy()  # (64, 128)
b2 = state["net.2.bias"].numpy()    # (64,)
W3 = state["net.4.weight"].numpy()  # (55, 64)
b3 = state["net.4.bias"].numpy()    # (55,)

print(f"Layer shapes: W1={W1.shape}, W2={W2.shape}, W3={W3.shape}")
print(f"Total params: {W1.size + b1.size + W2.size + b2.size + W3.size + b3.size:,}")

# Metadata
metadata = {
    "model_type": "mlp",
    "architecture": [D, 128, 64, NUM_WEIGHTS],
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "epochs_trained": epoch + 1,
    "n_train": len(train_pairs),
    "n_val": len(val_pairs),
    "n_features": D,
    "vocab_size": len(vocab),
    "val_quality_score": mlp_val_metrics["quality_score"],
    "val_avg_mouth_mae": mlp_val_metrics["avg_mouth_mae"],
    "dataset": os.path.basename(dataset_path),
}

# Save
output_path = "phoneme_mlp.npz"
np.savez_compressed(
    output_path,
    model_type=np.array("mlp", dtype="U"),
    W1=W1, b1=b1,
    W2=W2, b2=b2,
    W3=W3, b3=b3,
    vocab=np.array(vocab, dtype="U"),
    mean_duration=np.float32(mean_duration),
    metadata_json=np.array(json.dumps(metadata), dtype="U"),
)

file_size = os.path.getsize(output_path)
print(f"\nSaved: {output_path} ({file_size / 1024:.1f} KB)")
print(f"Quality: {mlp_val_metrics['quality_score']:.3f} (mouth MAE: {mlp_val_metrics['avg_mouth_mae']:.4f})")

print(f"\n--- Deployment ---")
print(f"Copy to production:")
print(f"  cp {output_path} <inference-project>/models/phoneme.npz")

if IN_COLAB:
    from google.colab import files
    files.download(output_path)
    print(f"\nDownloading {output_path}...")